## Imports

In [21]:
import sys
sys.path.insert(0, '../')

In [22]:
import math
import soundfile as sf
import numpy as np
from training.data_prep.audio_normalisation import AudioNorm

## Testing using audio from capuchinbird machine learning mini project
### Possible issues:
- Capuchinbird audio data has some that is stereo, need to convert to mono by taking only 1 audio channel.
- Sample rate is 48000 Hz instead of 44100 Hz but I think shouldn't be an issue for just testing segmentation

### Possible Bug Relating to segment = audio[start : start + window_samples]

I don't remember the exact issue or bug mentioned in the meeting on (16/07/2026) so I'm going off of what was written in the Notion AI Transcript:  
"Udhay flagged a potential bug: when indexing audio from start to start + window_samples, the index may exceed array bounds if the last segment is near the end of the file — Douglas was unsure; testing will confirm"  
  
I believe this shouldn't be an issue as if you slice past the end of a numpy array, numpy just returns the elements it can instead of throwing an error. 

In [37]:
array = np.array([1,2,3,4,5])
print(array[2:10])

[3 4 5]


### Functions used to test

In [23]:
def to_mono(file_name: str):
    audio_stereo, file_sr = sf.read(f'../test/data/Parsed_Capuchinbird_Clips/{str(file_name)}')
    print(f"Shape: {audio_stereo.shape}")
    if audio_stereo.ndim == 2:
        audio_mono = audio_stereo[:, 0]  # stereo, take left channel
    else:
        audio_mono = audio_stereo         # already mono
    print(f"Duration: {len(audio_mono)/file_sr:.2f}s, samples: {len(audio_mono)}")

    return audio_mono

In [24]:
def segment(audio_mono):
    segments = norm.segmentation(audio_mono)
    print(f"Number of segments: {len(segments)}")
    for i, seg in enumerate(segments):
        print(f"  Segment {i}: {len(seg)} samples ({len(seg)/norm.sampling_rate:.2f}s)")
    return segments

In [ ]:
def check_segment_count(audio_mono, segments):
    hop_samples = int(norm.hop_size * norm.sampling_rate) # convert hop size from seconds to number of samples
    expected = math.ceil(len(audio_mono) / hop_samples) # ceil because a partial window at the end still counts as a segment
    actual = len(segments)
    print(f"Expected segments: {expected}")
    print(f"Actual segments:   {actual}")
    print(f"Count correct: {expected == actual}")

In [ ]:
def check_overlap(segments):
    window_samples = int(norm.window_size * norm.sampling_rate) # full window size in samples
    hop_samples = int(norm.hop_size * norm.sampling_rate) # how far the window moved forward
    overlap = window_samples - hop_samples # the number of samples shared between consecutive segments
    match = np.array_equal(segments[0][hop_samples:], segments[1][:overlap]) # the tail of segment 0 should be identical to the head of segment 1
    print(f"Overlap between segment 0 and 1 correct: {match}")

In [ ]:
def check_padding(audio_mono, segments):
    window_samples = int(norm.window_size * norm.sampling_rate) # full window size in samples
    hop_samples = int(norm.hop_size * norm.sampling_rate) # hop size in samples
    last = segments[-1] # the last segment is the one most likely to be padded
    last_start = (len(segments) - 1) * hop_samples # sample index where the last window begins
    original_tail = audio_mono[last_start:] # the real audio remaining from that point
    pad_length = window_samples - len(original_tail) # how many zeros were added to fill the window
    if pad_length > 0:
        print(f"Padding amount: {pad_length} samples ({pad_length/norm.sampling_rate:.3f}s)")
        print(f"Ends with silence: {np.all(last[-pad_length:] == 0.0)}") # check the padded portion is all zeros
        print(f"Content matches original: {np.array_equal(last[:len(original_tail)], original_tail)}") # check the real audio wasn't modified
    else:
        print("Last segment was full length — no padding applied")

In [28]:
def run_all_checks(audio_mono, segments):
    print("=== Segment Count ===")
    check_segment_count(audio_mono, segments)
    print("\n=== Overlap ===")
    check_overlap(segments)
    print("\n=== Silence Padding ===")
    check_padding(audio_mono, segments)

In [29]:
def test_file(file_name):
    audio_mono = to_mono(file_name)
    segments = segment(audio_mono)
    run_all_checks(audio_mono, segments)

### Testing on files and checking results

In [30]:
file_name = "XC114131-0.wav"
test_file(file_name)

Shape: (120000, 2)
Duration: 2.50s, samples: 120000
Number of segments: 3
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 3
Actual segments:   3
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 100500 samples (2.279s)
Ends with silence: True
Content matches original: True


In [31]:
file_name = "XC114131-1.wav"
test_file(file_name)

Shape: (144000, 2)
Duration: 3.00s, samples: 144000
Number of segments: 4
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
  Segment 3: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 4
Actual segments:   4
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 120600 samples (2.735s)
Ends with silence: True
Content matches original: True


In [32]:
file_name = "XC3776-0.wav"
test_file(file_name)

Shape: (132300,)
Duration: 3.00s, samples: 132300
Number of segments: 3
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 3
Actual segments:   3
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 88200 samples (2.000s)
Ends with silence: True
Content matches original: True


In [33]:
file_name = "XC3776-1.wav"
test_file(file_name)

Shape: (132300,)
Duration: 3.00s, samples: 132300
Number of segments: 3
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 3
Actual segments:   3
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 88200 samples (2.000s)
Ends with silence: True
Content matches original: True


In [34]:
file_name = "XC201990-3.wav"
test_file(file_name)

Shape: (96000, 2)
Duration: 2.00s, samples: 96000
Number of segments: 3
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 3
Actual segments:   3
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 124500 samples (2.823s)
Ends with silence: True
Content matches original: True


In [35]:
file_name = "XC79965-3.wav"
test_file(file_name)

Shape: (220500,)
Duration: 5.00s, samples: 220500
Number of segments: 5
  Segment 0: 132300 samples (3.00s)
  Segment 1: 132300 samples (3.00s)
  Segment 2: 132300 samples (3.00s)
  Segment 3: 132300 samples (3.00s)
  Segment 4: 132300 samples (3.00s)
=== Segment Count ===
Expected segments: 5
Actual segments:   5
Count correct: True

=== Overlap ===
Overlap between segment 0 and 1 correct: True

=== Silence Padding ===
Padding amount: 88200 samples (2.000s)
Ends with silence: True
Content matches original: True
